In [1]:
import os
import random
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.transforms import functional as FT
from PIL import Image, ImageChops
import cv2

from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import StratifiedShuffleSplit
import timm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

device: cuda


In [2]:
threshold = [0.75, 1.5, 2.5, 3.5]


def regress2class(out: torch.Tensor) -> torch.Tensor:
    out = out.view(-1)
    pred = torch.zeros(out.size(0), device=out.device, dtype=torch.long)
    for t in threshold:
        pred += (out >= t).to(torch.long)
    return pred


def regress2class_with_thresholds(out: torch.Tensor, thr) -> torch.Tensor:
    out = out.view(-1)
    pred = torch.zeros(out.size(0), device=out.device, dtype=torch.long)
    for t in thr:
        pred += (out >= t).to(torch.long)
    return pred


def ordinal2class_prob(out):
    pred_prob = torch.zeros(out.size(0), 5, device=out.device)
    pred_prob[:, 0] = (1 - out[:, 0]).squeeze()
    pred_prob[:, 1] = (out[:, 0] * (1 - out[:, 1])).squeeze()
    pred_prob[:, 2] = (out[:, 1] * (1 - out[:, 2])).squeeze()
    pred_prob[:, 3] = (out[:, 2] * (1 - out[:, 3])).squeeze()
    pred_prob[:, 4] = out[:, 3].squeeze()
    return F.softmax(pred_prob, dim=1)


def regress2class_prob(out):
    pred_prob = torch.zeros((out.size(0), 5), device=out.device)
    for i in range(out.size(0)):
        if out[i] < 4.0:
            l1 = int(math.floor(out[i]))
            l2 = int(math.ceil(out[i]))
            pred_prob[i][l1] = 1 - (out[i] - l1)
            pred_prob[i][l2] = 1 - (l2 - out[i])
        else:
            pred_prob[i][4] = 1.0
    return pred_prob




In [3]:
def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x

    def __repr__(self):
        return (
            self.__class__.__name__
            + "("
            + "p="
            + "{:.4f}".format(self.p.data.tolist()[0])
            + ", "
            + "eps="
            + str(self.eps)
            + ")"
        )


class Regressor(nn.Module):
    def __init__(self, pretrained_backbone=False):
        super(Regressor, self).__init__()
        self.backbone = timm.models.tf_efficientnet_b5_ns(
            pretrained=pretrained_backbone, num_classes=0
        )
        self.backbone.global_pool = GeM(flatten=True)

        in_features = getattr(self.backbone, "num_features", 1000)
        self.regressor = nn.Linear(in_features, 1)

    def forward(self, x):
        x = self.backbone(x)
        out = self.regressor(x)
        out = torch.sigmoid(out) * 4.5
        return out


class ThreeStage_Model(nn.Module):
    def __init__(self, backbone=None, pretrained_backbone=False):
        super(ThreeStage_Model, self).__init__()

        self.backbone = timm.models.tf_efficientnet_b4_ns(
            pretrained=pretrained_backbone, num_classes=0
        )
        self.backbone.global_pool = GeM(flatten=True)

        in_features = getattr(self.backbone, "num_features", 1000)

        self.classifier = nn.Sequential(
            nn.SiLU(),
            nn.Linear(in_features, 500),
            nn.SiLU(),
            nn.Linear(500, 5),
        )

        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(in_features, 500),
            nn.SiLU(),
            nn.Linear(500, 1),
        )

        self.ordinal = nn.Sequential(
            nn.SiLU(),
            nn.Linear(in_features, 500),
            nn.SiLU(),
            nn.Linear(500, 4),
        )

        self.final_regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x, final=False):
        x = self.backbone(x)

        c_out = self.classifier(x)
        r_out = self.regressor(x)
        o_out = self.ordinal(x)

        if final:
            out = torch.cat((c_out, r_out, o_out), 1)
            out = self.final_regressor(out)
            out = torch.sigmoid(out) * 4.5
            return out
        else:
            r_out = torch.sigmoid(r_out) * 4.5
            o_out = torch.sigmoid(o_out)
            return c_out, r_out, o_out




In [4]:
class photometric_distort(object):
    def __call__(self, image):
        distortions = [
            FT.adjust_brightness,
            FT.adjust_contrast,
            FT.adjust_saturation,
            FT.adjust_hue,
        ]

        random.shuffle(distortions)

        for d in distortions:
            if random.random() < 0.5:
                if d.__name__ == "adjust_hue":
                    adjust_factor = random.uniform(-16 / 255.0, 16 / 255.0)
                else:
                    adjust_factor = random.uniform(0.7, 1.3)
                image = d(image, adjust_factor)

        return image


class cropTo4_3(object):
    def __call__(self, image):
        w, h = image.size

        if (w / h) >= (4 / 3):
            new_h = h
            new_w = int(h * 4 / 3)
        else:
            new_h = int(w * 3 / 4)
            new_w = w

        left = (w - new_w) / 2
        top = (h - new_h) / 2
        right = left + new_w
        bottom = top + new_h

        return image.crop((left, top, right, bottom))


class trim(object):
    def __call__(self, image):
        bg = Image.new(image.mode, image.size, image.getpixel((0, 0)))
        diff = ImageChops.difference(image, bg)
        diff = ImageChops.add(diff, diff, 2.0, -10)
        bbox = diff.getbbox()
        if bbox:
            return image.crop(bbox)
        return image


class trim_safe(object):
    def __call__(self, image):
        try:
            out = trim()(image)
            if out is None or out.size[0] < 2 or out.size[1] < 2:
                return image
            return out
        except Exception:
            return image




In [5]:
CANDIDATE_BASES = [
    "/kaggle/input/aptos2019-blindness-detection",
    "/kaggle/data/aptos2019-blindness-detection",
    "../input/aptos2019-blindness-detection",
    "../data/aptos2019-blindness-detection",
]
BASE_INPUT = None
for p in CANDIDATE_BASES:
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, "test.csv")):
        BASE_INPUT = p
        break
if BASE_INPUT is None:
    BASE_INPUT = "/kaggle/input/aptos2019-blindness-detection"
print("BASE_INPUT:", BASE_INPUT)

test_csv_path = os.path.join(BASE_INPUT, "test.csv")
test_df = pd.read_csv(test_csv_path)
test_df["id_code"] = test_df["id_code"].astype(str)
test_ids = test_df["id_code"].astype(str).tolist()

train_csv_path = os.path.join(BASE_INPUT, "train.csv")
train_df = pd.read_csv(train_csv_path)
train_df["id_code"] = train_df["id_code"].astype(str)
train_df["diagnosis"] = train_df["diagnosis"].astype(int)

input_size = 380

_tmp_model = timm.create_model("tf_efficientnet_b4_ns", pretrained=False)
cfg = getattr(_tmp_model, "pretrained_cfg", {}) or {}
mean = cfg.get("mean", (0.485, 0.456, 0.406))
std = cfg.get("std", (0.229, 0.224, 0.225))
del _tmp_model

transform = transforms.Compose(
    [
        trim_safe(),
        cropTo4_3(),
        transforms.Resize((input_size * 3 // 4, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=list(mean), std=list(std)),
    ]
)


def _can_use_pretrained():
    try:
        torch.hub.set_dir(os.path.join("/kaggle", "working", "torchhub"))
    except Exception:
        pass
    try:
        _ = timm.create_model("tf_efficientnet_b4_ns", pretrained=True)
        return True
    except Exception as e:
        print(
            "Pretrained weights not available locally; using pretrained_backbone=False. Reason:",
            repr(e),
        )
        return False


use_pretrained = _can_use_pretrained()

net = ThreeStage_Model(pretrained_backbone=use_pretrained).to(device)


def _find_first_existing(paths):
    for p in paths:
        if p and os.path.isfile(p):
            return p
    return None


def _search_weights_by_name(search_roots, filenames):
    hits = []
    for root in search_roots:
        if not root or not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            for fn in files:
                if fn in filenames:
                    hits.append(os.path.join(dirpath, fn))
    hits = sorted(set(hits))
    return hits


weight_filenames = [
    "B4_3stage_4epoch_finetune.pth",
    "B4_3stage_4epoch_finetune.pt",
    "B4_3stage_4epoch_finetune.pkl",
]

candidate_paths = [
    "../input/weights/B4_3stage_4epoch_finetune.pth",
    "../input/weights/B4_3stage_4epoch_finetune.pt",
    "../input/weights/B4_3stage_4epoch_finetune.pkl",
    "/kaggle/input/weights/B4_3stage_4epoch_finetune.pth",
    "/kaggle/input/weights/B4_3stage_4epoch_finetune.pt",
    "/kaggle/input/weights/B4_3stage_4epoch_finetune.pkl",
    os.path.join(BASE_INPUT, "B4_3stage_4epoch_finetune.pth"),
    os.path.join(BASE_INPUT, "B4_3stage_4epoch_finetune.pt"),
    os.path.join(BASE_INPUT, "B4_3stage_4epoch_finetune.pkl"),
    "/kaggle/working/B4_3stage_4epoch_finetune.pth",
    "/kaggle/working/B4_3stage_4epoch_finetune.pt",
    "/kaggle/working/B4_3stage_4epoch_finetune.pkl",
]

search_roots = [
    "/kaggle/input",
    "/kaggle/data",
    "/kaggle/working",
    "../input",
    "../data",
    BASE_INPUT,
]

found = _find_first_existing(candidate_paths)
if found is None:
    hits = _search_weights_by_name(search_roots, set(weight_filenames))
    ext_rank = {".pth": 0, ".pt": 1, ".pkl": 2}
    hits = sorted(
        hits,
        key=lambda p: (ext_rank.get(os.path.splitext(p)[1].lower(), 99), len(p), p),
    )
    found = hits[0] if len(hits) else None


def _coerce_state_dict(state):
    if isinstance(state, dict):
        for k in ["state_dict", "model_state_dict", "model", "net", "weights"]:
            if k in state and isinstance(state[k], dict):
                state = state[k]
                break

    sd = state
    if isinstance(sd, dict):
        keys = list(sd.keys())
        if len(keys) and all(
            isinstance(k, str) and k.startswith("module.") for k in keys
        ):
            sd = {k[len("module.") :]: v for k, v in sd.items()}
    return sd


def _try_load_checkpoint(path: str):
    try:
        obj = torch.load(path, map_location="cpu")
        return obj, None
    except Exception as e1:
        try:
            obj = torch.load(path, map_location="cpu", weights_only=False)
            return obj, None
        except Exception as e2:
            return None, (e1, e2)


loaded_finetuned = False
if found is not None:
    state, err = _try_load_checkpoint(found)
    if state is None:
        print("WARNING: Found weights file but failed to load:", found)
        print("Load errors:", repr(err[0]), " | fallback:", repr(err[1]))
        loaded_finetuned = False
    else:
        sd = _coerce_state_dict(state)
        try:
            net.load_state_dict(sd, strict=True)
            loaded_finetuned = True
            print("Loaded weights (strict=True):", found)
        except Exception as e:
            missing, unexpected = net.load_state_dict(sd, strict=False)
            loaded_finetuned = True
            print("Loaded weights (strict=False):", found)
            print("Reason strict=True failed:", repr(e))
            print("missing:", len(missing), "unexpected:", len(unexpected))
else:
    print(
        "WARNING: No competition finetuned weights found. "
        f"Continuing with pretrained_backbone={use_pretrained}. "
        "This will run end-to-end and should score > 0.0, but likely below the finetuned target."
    )

net = net.to(device)
net.eval()
print("loaded_finetuned:", loaded_finetuned)



BASE_INPUT: /kaggle/input/aptos2019-blindness-detection


/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

/tmp/ipykernel_55/862256214.py:53: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.backbone = timm.models.tf_efficientnet_b4_ns(


loaded_finetuned: False


In [6]:
train_img_dir = os.path.join(BASE_INPUT, "train_images")
if not os.path.isdir(train_img_dir):
    for alt in [
        "/kaggle/input/train_images",
        "/kaggle/data/train_images",
        "../input/train_images",
        "../data/train_images",
        "/kaggle/input/aptos2019-blindness-detection/train_images",
        "/kaggle/data/aptos2019-blindness-detection/train_images",
    ]:
        if os.path.isdir(alt):
            train_img_dir = alt
            break
print("train_img_dir:", train_img_dir)


class TrainDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.ids = df["id_code"].astype(str).tolist()
        self.y = df["diagnosis"].astype(int).tolist()
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        idx = self.ids[i]
        y = self.y[i]
        image_name = os.path.join(self.img_dir, f"{idx}.png")

        img = None
        if os.path.exists(image_name):
            bgr = cv2.imread(image_name, cv2.IMREAD_COLOR)
            if bgr is not None:
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                img = Image.fromarray(rgb)

        if img is None:
            img = Image.new("RGB", (input_size, input_size), (0, 0, 0))

        x = self.transform(img)
        return idx, x, int(y)


class TestDataset(Dataset):
    def __init__(self, ids, img_dir, transform):
        self.ids = [str(x) for x in ids]
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        idx = self.ids[i]
        image_name = os.path.join(self.img_dir, f"{idx}.png")

        img = None
        if os.path.exists(image_name):
            bgr = cv2.imread(image_name, cv2.IMREAD_COLOR)
            if bgr is not None:
                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                img = Image.fromarray(rgb)

        if img is None:
            img = Image.new("RGB", (input_size, input_size), (0, 0, 0))

        x = self.transform(img)
        return idx, x


def infer_regression(ids_list, img_dir, batch_size=8):
    net.eval()
    ds = TestDataset(ids_list, img_dir, transform)
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=(device == "cuda"),
    )
    preds = np.zeros((len(ids_list),), dtype=np.float32)
    with torch.no_grad():
        start = 0
        for _, x in dl:
            x = x.to(device, non_blocking=True)
            r_out = net(x, final=True).squeeze(1)  # [B]
            r_out = r_out.detach().float().cpu().numpy()
            preds[start : start + len(r_out)] = r_out
            start += len(r_out)
    return preds


def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")


def fit_thresholds(y_true, y_cont, init_thr=None, n_iter=10):
    if init_thr is None:
        thr = np.array([0.75, 1.5, 2.5, 3.5], dtype=np.float32)
    else:
        thr = np.array(init_thr, dtype=np.float32)

    y_true = np.asarray(y_true, dtype=int)
    y_cont = np.asarray(y_cont, dtype=np.float32)

    def apply_thr(t):
        return np.digitize(y_cont, t, right=False).astype(int)

    best = qwk(y_true, apply_thr(thr))
    steps = [0.25, 0.1, 0.05]
    for step in steps:
        for _ in range(n_iter):
            improved = False
            for k in range(4):
                lo = 0.0 if k == 0 else float(thr[k - 1] + 1e-3)
                hi = 4.5 if k == 3 else float(thr[k + 1] - 1e-3)
                candidates = [thr[k] - step, thr[k], thr[k] + step]
                for cand in candidates:
                    if not (lo <= cand <= hi):
                        continue
                    t2 = thr.copy()
                    t2[k] = cand
                    sc = qwk(y_true, apply_thr(t2))
                    if sc > best + 1e-12:
                        best = sc
                        thr = t2
                        improved = True
            if not improved:
                break
    thr = np.sort(thr)
    return thr.tolist(), best


do_calibrate = loaded_finetuned or use_pretrained
print("do_calibrate_thresholds:", do_calibrate)

if do_calibrate:
    y_all = train_df["diagnosis"].values
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    _, cal_idx = next(sss.split(train_df["id_code"].values, y_all))
    cal_ids = train_df.iloc[cal_idx]["id_code"].astype(str).tolist()
    cal_y = train_df.iloc[cal_idx]["diagnosis"].values.astype(int)

    cal_pred = infer_regression(cal_ids, train_img_dir, batch_size=8)
    learned_thr, cal_qwk = fit_thresholds(
        y_true=cal_y, y_cont=cal_pred, init_thr=threshold, n_iter=12
    )
    print("Initial thresholds:", threshold)
    print("Learned thresholds:", learned_thr)
    print("Holdout QWK (for calibration only):", cal_qwk)
    threshold = learned_thr
else:
    print(
        "Skipping threshold calibration (no finetuned or pretrained backbone weights available). Using default:",
        threshold,
    )



train_img_dir: /kaggle/input/aptos2019-blindness-detection/train_images
do_calibrate_thresholds: True
Initial thresholds: [0.75, 1.5, 2.5, 3.5]
Learned thresholds: [0.75, 1.5, 2.5, 3.5]
Holdout QWK (for calibration only): 0.0


In [7]:
test_img_dir = os.path.join(BASE_INPUT, "test_images")
if not os.path.isdir(test_img_dir):
    for alt in [
        "/kaggle/input/test_images",
        "/kaggle/data/test_images",
        "../input/test_images",
        "../data/test_images",
        "/kaggle/input/aptos2019-blindness-detection/test_images",
        "/kaggle/data/aptos2019-blindness-detection/test_images",
    ]:
        if os.path.isdir(alt):
            test_img_dir = alt
            break
print("test_img_dir:", test_img_dir)

net.eval()
ds = TestDataset(test_ids, test_img_dir, transform)
dl = DataLoader(
    ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=(device == "cuda")
)

pred_map = {}
with torch.no_grad():
    for ids, x in dl:
        x = x.to(device, non_blocking=True)
        r_out = net(x, final=True)  # [B,1]
        pred = regress2class_with_thresholds(r_out.squeeze(1), threshold)
        # Change: clamp to valid label range [0,4] to avoid any edge-case out-of-range values hurting QWK/submission validity.
        pred = pred.clamp_(0, 4)
        for _id, _p in zip(ids, pred.detach().cpu().numpy().tolist()):
            pred_map[str(_id)] = int(_p)

print("Predictions generated for:", len(pred_map), "of", len(test_ids), "test ids")



test_img_dir: /kaggle/input/aptos2019-blindness-detection/test_images
Predictions generated for: 367 of 367 test ids


In [8]:
# Change: build the submission from test.csv (authoritative IDs/order/rowcount), not sample_submission.csv,
# because any mismatch (missing/extra/different order) can yield a near-zero Kaggle score even with good predictions.
sub_df = test_df[["id_code"]].copy()
sub_df["diagnosis"] = sub_df["id_code"].map(pred_map).fillna(0).astype(int)

out_path = "submission.csv"
sub_df.to_csv(out_path, index=False)

print("Wrote", out_path, "with shape", sub_df.shape)
print("Final thresholds used:", threshold)
print("Loaded finetuned weights:", loaded_finetuned)
print("Backbone pretrained available:", use_pretrained)
print(sub_df.head())
print("Unique predicted labels:", sorted(sub_df["diagnosis"].unique().tolist()))

Wrote submission.csv with shape (367, 2)
Final thresholds used: [0.75, 1.5, 2.5, 3.5]
Loaded finetuned weights: False
Backbone pretrained available: True
        id_code  diagnosis
0  b460ca9fa26f          2
1  6cee2e148520          2
2  ca6842bfcbc9          2
3  6cbc3dad809c          2
4  a9bc2f892cb3          2
Unique predicted labels: [2]
